# Roof segmentation

Same question as the population fits: does the network beat a null model you can write down without it.

25 labeled 256x256 tiles, 5 unlabeled. A pixel is a Bernoulli trial. H0 is the mean roof rate of the training masks. H1 is a residual U-Net. The submitted masks are the five unlabeled ids.


In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

SEED = 41
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"seed {SEED}  torch {torch.__version__}  cuda {torch.cuda.is_available()}")

plt.rcParams.update({"font.family": "Serif", "font.size": 10})
plt.style.use("tableau-colorblind10")


## What is actually in the PNG

Images are RGBA. Alpha 0 means a missing tile, and RGB is 0 there too. That is not a black roof. Labels are grayscale because the roof edge was anti-aliased. The likelihood uses the hard state `gray > 127`. Threshold 0.5 is only for the IoU and for the PNG that gets sent.


In [ ]:
from roofseg.config import ExperimentConfig
from roofseg.data_loader import discrete_mask, list_ids, load_rgb, occupancy, split_ids
cfg = ExperimentConfig()

image_dir = Path("data/images")
label_dir = Path("data/labels")
labeled = [i for i in list_ids(image_dir) if (label_dir / f"{i}.png").exists()]
unlabeled = [i for i in list_ids(image_dir) if i not in set(labeled)]
train_ids, val_ids = split_ids(labeled, cfg.val_fraction, SEED)
prior = occupancy(image_dir, label_dir, train_ids, cfg)
print("train", train_ids)
print("val  ", val_ids)
print("test ", unlabeled)
print(f"H0 roof rate p0 = {prior:.4f}")

show = ["121", "270", "274"]
fig, axes = plt.subplots(len(show), 2, figsize=(5.2, 7.2), dpi=150)
for row, image_id in enumerate(show):
    rgb, valid = load_rgb(image_dir / f"{image_id}.png", cfg.valid_alpha)
    mask = discrete_mask(label_dir / f"{image_id}.png", cfg.mask_threshold)
    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title(image_id)
    axes[row, 1].imshow(mask, cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title("roof state")
    for ax in axes[row]:
        ax.set_xticks([])
        ax.set_yticks([])
axes[0, 0].set_ylabel("image")
fig.tight_layout()
fig.savefig("outputs/label_examples.png", dpi=200)
plt.show()


## Baseline

$p_0$ is the roof fraction on the training photos, counted only where the satellite tile is present. The same number is painted everywhere. On a photo left out of the fit, the network has to put more probability on the real roof pixels than that flat guess does.

$$
\log\mathcal{L} = \sum_{i \in \mathrm{observed}} \big[ y_i \log p_i + (1-y_i)\log(1-p_i) \big]
$$

The table below is $2\Delta\log\mathcal{L}$ on selection.pt. The five masks come from final.pt. That file was not scored.

About 15% of a tile is roof, so a plain sum over pixels is mostly background. Soft Dice gives the roof outline the same weight as the rest of the tile while training. It is not part of the sum above.


In [ ]:
import json
metrics_path = Path("outputs/metrics.json")
if not metrics_path.exists():
    raise SystemExit("no outputs/metrics.json yet. run: python -m roofseg.execute_experiment")
report = json.loads(metrics_path.read_text())
rows = report["per_image"]
print(f"selected epoch {report['best_epoch']}  val loss {report['best_val_loss']:.4f}")
print(f"{'id':>6} {'logL_H0':>12} {'logL_H1':>12} {'delta':>10} {'TS':>10} {'iou':>8}")
for r in rows:
    print(f"{r['id']:>6} {r['logL_H0']:12.1f} {r['logL_H1']:12.1f} {r['delta_logL']:10.1f} {r['TS']:10.1f} {r['iou']:8.3f}")

fig, ax = plt.subplots(figsize=(5, 3), dpi=300)
ax.bar([r["id"] for r in rows], [r["delta_logL"] for r in rows], edgecolor="black")
ax.set_xlabel("hold-out tile")
ax.set_ylabel(r"$\Delta\log\mathcal{L}$ (nats)")
ax.set_title(r"Network vs constant roof rate")
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig("outputs/holdout_delta_logl.png", dpi=300)
plt.show()


## Tile 274

About 1300 nats better than the flat guess, and the drawing at probability 0.5 is empty. The probabilities can sit a little above $p_0$ on the roof and a little below it on the background, which helps the sum, and still never reach one half. Everything is then drawn as background. The roof signal is there, spread out, and too weak for the cutoff.


## Tile 278

IoU is about 0.08, and the likelihood is about 58000 nats worse than the flat guess. A few pixels are nearly certain and wrong: roof where there is none, or background where the roof is. Those pixels dominate the sum. The small overlap only means some pixels above 0.5 landed on the roof. It does not say how sure the model was on the rest of the tile.


## Test masks

`outputs/predictions/{535,537,539,551,553}.png` are the files to send. 255 on the roof, 0 elsewhere, missing tiles forced to 0. The red overlays are only for looking.


In [ ]:
pred_dir = Path("outputs/predictions")
overlay_dir = Path("outputs/overlays/test")
if not pred_dir.exists() or not any(pred_dir.glob("*.png")):
    raise SystemExit("no predictions yet. run: python -m roofseg.execute_experiment")

fig, axes = plt.subplots(1, len(unlabeled), figsize=(12, 2.6), dpi=150)
for ax, image_id in zip(axes, unlabeled):
    overlay = overlay_dir / f"{image_id}.png"
    img = Image.open(overlay if overlay.exists() else pred_dir / f"{image_id}.png")
    ax.imshow(img)
    ax.set_title(image_id)
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
plt.show()
